# 05 — Temporal & spend analysis

The actual *findings* notebook. Up to this point everything's been infrastructure — preprocessing (01), classification (02), topic modelling (03). This notebook leverages those labels to answer the questions the report cares about.

**Headline narrative**: political advertising is bimodal. Candidate and party-org spending spikes 10×–50× above baseline in the weeks before election day and collapses immediately after. NGO and government advertising stays roughly flat across the same window. The classifier built in notebooks 02–03 separates the two ecosystems empirically — this notebook quantifies the gap.

**Window**: 6 months before to 6 months after the 21 May 2022 federal election (set in [01_data_loading.ipynb](01_data_loading.ipynb)).

## 1. Setup

Spark session with the same parquet-committer override as notebooks 02/03. Load v3 parquet — the corpus with `political_party`, `match_type`, `topic_id`, `topic_label`, and `category` columns attached. One DataFrame, all analyses below.

## 2. Headline finding — bimodal ad ecosystem

**Question**: how differently do election-driven and steady-state advertisers behave temporally?

**Approach**: weekly spend and weekly ad-volume, stacked by `match_type`, with the election date as a vertical marker. Two views (spend on the left, count on the right) to disentangle outlier-driven from volume-driven signals.

**Expected shape**: `candidate` + `party_org` spike sharply in Apr–May 2022 and collapse to near-zero by mid-June. `government` and `unclassified` (NGO/advocacy) stay roughly constant.

**Election-surge ratio**: per category, compute `peak_week_spend / median_other_week_spend`. Tabulate. This single metric quantifies the bimodal claim.

## 3. Top advertisers — absolute and proportional

**Question**: who held the megaphone, and when?

**Approach**: take the top 10–12 bylines by total `spend_mid` across the window. Two side-by-side timeseries:

- **Chart A — absolute spend**: weekly spend per byline, lines coloured by `match_type`. Shows raw spending pattern. UAP's election concentration should be visible; Greenpeace's flat profile should contrast.
- **Chart B — proportional share**: weekly spend share within the top-N, stacked area to 100%. Shows compositional dominance. UAP grabs majority share for ~6 weeks; NGOs collectively dominate the rest of the window.

Together: A tells you the absolute magnitudes, B tells you who was loudest at each moment.

## 4. Which topics survive the election

**Question**: which themes are election-driven and which are evergreen?

**Approach**: weekly ad count per `topic_label`, normalised to each topic's median weekly count. Plot a heatmap (topics × weeks) or a small-multiples grid showing each topic's temporal profile.

**Expected pattern**:

- *Election-driven* (sharp pre-election spike, post-election collapse): `anti_morrison_climate`, `election_independents`, `anti_labor_plastic_mixed`, `transport_fuel_costs`, `funding_advocacy` (AEU election push).
- *Evergreen* (relatively flat): `environment_donate`, `ocean_treaty_global`, `charity_violence_women`, `humanitarian_refugees`, `medicines_costs`.
- *Mixed* (broad activity but moderate election lift): `climate_climb_event`, `early_childhood_climate`, `wildlife_environment`.

The split lets us empirically separate "campaign messaging" from "issue advocacy".

## 5. Advertiser persistence — who's still here in November 2022?

**Question**: which advertisers maintained activity after the election ended?

**Approach**: split the window in half (Nov 2021 – May 2022 = pre, May 2022 – Nov 2022 = post). For each byline, compute `post_ads / pre_ads` ratio.

- Ratio ≈ 1.0 → steady advertiser (most NGOs, government).
- Ratio << 1.0 → election-only advertiser (party central offices, individual candidates, attack-ad funders).
- Ratio > 1.0 → post-election ramp (uncommon — maybe newly-elected MPs, or NGOs that started after the election).

Tabulate the top-10 most-persistent and top-10 least-persistent advertisers above a minimum-spend threshold.

## 6. Spend distribution — by category and outlier behaviour

**Question**: how does spend per ad differ across categories?

**Approach**: log-scale boxplot or violin of `spend_mid` by `match_type` (and possibly by `topic_label` for the residual). Reveals which categories have a long tail of mega-budget ads. UAP/candidate ads are likely to be high-mean, high-variance; NGO advocacy is likely low-mean, low-variance, with occasional Greenpeace big-ticket campaigns as outliers.

Side check: do the long-tail outliers within `unclassified` (the residual political-looking advertisers) look more like Greenpeace-style high-stakes or more like commercial noise?

## 7. Discussion & main message

Three findings, woven into the report's discussion section:

1. **The ecosystem is bimodal.** A 10×–50× election surge in candidate/party-org spending against a flat NGO/government baseline. Captures the headline.
2. **Topic-level evidence corroborates the bimodal split.** Anti-Morrison and election-independents themes are election-only; conservation, oceans, humanitarian, and medicines themes are evergreen.
3. **Most loud political advertising vanishes after the election; most advocacy persists.** Quantified by advertiser-persistence ratios.

**Main message for non-technical stakeholders**: Australian political advertising on Facebook in 2022 was *two separate phenomena* — a transient campaign-period spike funded by parties and individual candidates (and dominated in spend by UAP), layered over an ongoing background of NGO and advocacy activity that runs continuously regardless of election cycles. Analysing them as a single "political advertising" category misses the structural distinction.

## Status

**Stub only — implementation pending the post-election rerun of notebooks 01 → 02 → 03.** Once v3 is rebuilt with the full 12-month window, each section above gets a code cell with the Spark aggregation + pandas/matplotlib visualisation.